## Notebook Description: Agricultural Data Processing and Analysis
This Colab notebook outlines a detailed data processing pipeline for agricultural research, focusing on gas and nutrition measurements. The workflow begins with loading datasets from Google Drive, followed by robust data cleaning and standardization of identifiers using functions such as `clean_standardize_ids()`. It involves converting data types, filtering records based on criteria like functional group and batch, and aggregating numerical data (e.g., $ch4\_8h\_ml$, $ch4\_24h\_ml$, $tddm$, $dm\_percentage$, $ash\_dm$, $om\_percentage$, $pc\_percentage\_dm$, $adf\_percentage\_dm$, $ndf\_percentage\_dm$) to calculate averages and replicate counts using a custom `mean_with_replicates()` function. The processed data is then compiled and exported into several specialized CSV files, serving diverse analytical needs including training sets for a Shiny App, gas dashboards, metabolomics primary traits, and analyses for grasses breeding and Stylosanthes gene bank.

## i) Import Libraries

In [134]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [135]:
import re
import pandas as pd

## ii) Data Cleaning Functions

In [136]:
# Function to remove the first row (duplicated original column names)
def remove_first_row(df):
    return df.iloc[1:, :].copy()

# Usage:
# subset_1_information_samples = remove_first_row(subset_1_information_samples)





# Function to clean and standardize lab/sample IDs to FXX-XXXX format
def clean_lab_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'\s+', '-', regex=True)                 # Replace spaces with -
        .str.replace(r'^(F\d{2})(\d+)', r'\1-\2', regex=True) # Format FXX-XXXX
    )


# Usage:
# subset_1_information_samples['10_ciat_lab_id'] = clean_lab_ids(subset_1_information_samples['10_ciat_lab_id'])



# Function to clean and standardize Gene Bank - Breeding program IDs
def clean_standardize_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'-1$', '', regex=True)                   # Remove trailing -1
        .str.strip()                                           # Remove leading/trailing spaces
        .str.replace(r'[_\s]+', '-', regex=True)               # Replace spaces/underscores with -
        .str.replace(r'([A-Za-z])(\d)', r'\1-\2', regex=True)  # Letter followed by number
        .str.replace(r'(\d)([A-Za-z])', r'\1-\2', regex=True)  # Number followed by letter
        .str.replace(r'-+', '-', regex=True)                   # Remove repeated -
        .str.replace(r'^-|-$', '', regex=True)                 # Remove leading/trailing -
        .str.replace('ABC-', 'CIAT-', regex=False)             # Replace id's strings 'ABC' with 'CIAT'
    )

    # subset_1_information_samples['10_gene_bank_breeding_program_id'] = clean_standardize_ids(subset_1_information_samples['10_gene_bank_breeding_program_id'])


# 1.0 Shiny App Training Set

 Requested by: Khaled Al-Sham'aa, ICARDA

 Date: 2026_05_19

In [137]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,3,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,24.237242,227.8476704,37.051645,65.414752,NaN,"1,2,3",NaN,NaN,NaN,NaN
1,1,4,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,24.362056,270.0095702,38.185077,63.799940,NaN,"1,2,3",NaN,NaN,NaN,NaN
2,1,5,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,23.863939,249.4607864,37.088353,64.343484,NaN,"1,2,3",NaN,NaN,NaN,NaN
3,1,6,Genetic bank,F24-3417,CIAT-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,25.091864,175.9559634,41.714068,60.152044,NaN,NaN,NaN,NaN,NaN,NaN
4,1,7,Genetic bank,F24-3417,CIAT-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,24.918358,149.5197103,39.832832,62.557335,NaN,NaN,Inicio embolo duro,Inicio embolo duro,NaN,NaN


In [138]:
# Filter by functional group 'Grass'
subsets_gas_grass = subsets_gas[subsets_gas['functional_group'] == 'Grass']
subsets_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,13.216913,194.6816788,27.337315,48.347519,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,13.472449,117.6219867,25.413695,53.012554,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,14.131561,195.7779286,28.971955,48.776690,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,14.823109,89.5309162,25.377242,58.411031,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.808292,110.6977836,30.734726,57.941924,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN


In [139]:
# Filter by subset '2'
subset_2_gas_grass = subsets_gas_grass[subsets_gas_grass['subset'] == 2]
subset_2_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
2823,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Eliminado DMD,NaN,NaN
2824,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,NaN,NaN,NaN,NaN,NaN,"30,31,32",NaN,Eliminado DMD,NaN,NaN
2825,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,NaN,NaN,NaN,NaN,NaN,"33,34,35",NaN,Eliminado DMD,NaN,NaN
2826,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,NaN,NaN,NaN,NaN,NaN,"36,37,38",NaN,Eliminado DMD,NaN,NaN
2827,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,NaN,NaN,NaN,NaN,NaN,"42,43,47",NaN,Eliminado DMD,NaN,NaN


In [140]:
subset_2_gas_grass.batch.unique()

array([34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 59, 60, 61])

In [141]:
# Filter by batch =< 40
subset_2_gas_grass_batchs = subset_2_gas_grass[subset_2_gas_grass['batch'] >= 50]
subset_2_gas_grass_batchs.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
4965,2,1701,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,23.847552,34.19146976,43.344262,55.018936,NaN,"36,37,38",NaN,NaN,NaN,NaN
4966,2,1702,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,24.175901,35.00485836,43.501476,55.574898,NaN,"36,37,38",NaN,NaN,NaN,NaN
4967,2,1703,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,24.464359,35.25395301,43.886070,55.745158,NaN,"36,37,38",NaN,NaN,NaN,NaN
4968,2,1704,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,24.174256,42.1835579,39.267646,61.562784,NaN,"36,37,38",NaN,NaN,NaN,NaN
4969,2,1705,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,25.486246,36.28321101,44.216137,57.640147,NaN,"36,37,38",NaN,NaN,NaN,NaN


In [142]:
#subset_2_gas_grass_batchs.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/ciat_gas_for_shiny_app_and_blues.csv', index = None)

# 2.0 Gas Dashboard of Subset 4

Requested by: Alejandra Marín

Date: 2026_05_22

In [143]:
dashboard_small = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_small.csv')
dashboard_small.head()

,subset,id_lab,id,tax_name,functional_group,ch4_8h_percentage,ch4_24h_percentage,methane_intensity,tddm
0,1,F25-0769,AMC-ICARDA-156302,Trifolium repens,Herbaceous_legumes,15.97,16.37,43.67,72.00
1,1,F25-0770,AMC-ICARDA-165072,Vicia tenuifolia,NaN,14.37,18.77,56.12,56.78
2,1,F25-0773,AMC-ICARDA-168125,Lathyrus sylvestris,NaN,15.53,17.77,55.10,42.32
3,1,F24-3469,CIAT-10647,Stylosanthes scabra,Herbaceous_legumes,15.90,22.28,48.31,67.34
4,1,F24-3470,CIAT-11194,Stylosanthes hamata,Herbaceous_legumes,15.17,20.65,46.12,59.25


In [144]:
# Filter data of subset 4
dashboard_small_subset_4 = dashboard_small[dashboard_small['subset'] == 4]

In [145]:
# Order by id
def natural_sort_key(s):
    """Splits a string into alphanumeric components and converts numbers to integers for natural sorting."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', str(s))]

# Apply natural sort to the 'id' column
dashboard_small_subset_4_natural_sorted = dashboard_small_subset_4.sort_values(
    by='id',
    key=lambda col: col.apply(natural_sort_key)
).reset_index(drop=True)

display(dashboard_small_subset_4_natural_sorted.head())

,subset,id_lab,id,tax_name,functional_group,ch4_8h_percentage,ch4_24h_percentage,methane_intensity,tddm
0,4,F25-2640,Cayman-Br-02-1752,Urochloa interespecifico,Grass,12.99,13.71,50.17,50.22
1,4,F25-2574,CIAT-326,Desmodium scorpiurus,Herbaceous_legumes,13.86,16.66,55.45,39.09
2,4,F25-2575,CIAT-415,Vigna radiata,Herbaceous_legumes,11.22,15.38,43.25,51.39
3,4,F25-2576,CIAT-416,Vigna radiata,Herbaceous_legumes,12.38,16.84,50.26,56.45
4,4,F25-2577,CIAT-517,Macroptilium atropurpureum,Herbaceous_legumes,11.58,14.88,52.88,37.99


In [146]:
#dashboard_small_subset_4_natural_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/subset_4_gas_dashboard_small.csv', index = None)

# 3.0 Grasses Gas Complete

Requested by: Claudia Perea

Date: 2026_05_26

In [147]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
0,1,3,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,24.237242,227.8476704,37.051645,65.414752,NaN,"1,2,3",NaN,NaN,NaN,NaN
1,1,4,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,24.362056,270.0095702,38.185077,63.799940,NaN,"1,2,3",NaN,NaN,NaN,NaN
2,1,5,Genetic bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,23.863939,249.4607864,37.088353,64.343484,NaN,"1,2,3",NaN,NaN,NaN,NaN
3,1,6,Genetic bank,F24-3417,CIAT-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,25.091864,175.9559634,41.714068,60.152044,NaN,NaN,NaN,NaN,NaN,NaN
4,1,7,Genetic bank,F24-3417,CIAT-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,24.918358,149.5197103,39.832832,62.557335,NaN,NaN,Inicio embolo duro,Inicio embolo duro,NaN,NaN


In [148]:
# Filter by functional group 'Grass'
grasses = subsets_gas[subsets_gas['functional_group'] == 'Grass']
grasses.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,13.216913,194.6816788,27.337315,48.347519,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,13.472449,117.6219867,25.413695,53.012554,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,14.131561,195.7779286,28.971955,48.776690,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,14.823109,89.5309162,25.377242,58.411031,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.808292,110.6977836,30.734726,57.941924,Ensayo interno,NaN,"Syringe with Bag, Difficul in remove bubble","Syringe with Bag, Difficul in remove bubble",NaN,NaN


In [149]:
#grasses.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/grasses.csv', index = None)

# 4.0 Primary Traits for Metabolomics

Requested by: Jenny Gallo

Date: 2026_06_03

In [150]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/requested_jenny.csv')
data = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_complete.csv')

In [151]:
requested = remove_first_row(requested)
requested.head(50)

,approach,gene_bank_breeding_id,tax_name,functional_group,ch4_intensity_ml_g_tddm,tddm_percentage,ch4_intensity_decrease,ch4_percentage_8h,ch4_percentage_24h,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
1,Approach 1 (a),CIAT-772,Clitoria ternatea,Climber,37.9,65.1,41%,13.45555556,14.98567864,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Approach 1 (a),CIAT-19213,Clitoria ternatea,Climber,39.1,63.7,39%,14.98888889,16.08929282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Approach 2,CIAT-9434,Clitoria ternatea,Climber,46.51,51.78,33%,14.36666667,15.85424307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Approach 2,CIAT-955,Clitoria ternatea,Climber,45.45,59.65,30%,12.82222222,14.54174742,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Approach 4,CIAT-9432,Clitoria ternatea,Climber,57.24,50.87,11%,17.36,18.90480266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Approach 4,CIAT-18447,Clitoria ternatea,Climber,53.0,55.3,18%,14.51111111,16.92415425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Approach 1 (a),CIAT-7317,Canavalia sp.,Climber,33.0,69.5,49%,14.86111111,16.36328667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Approach 1 (a),CIAT-8719,Canavalia sp.,Climber,35.8,69.0,45%,14.66388889,16.92290927,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Approach 2,CIAT-20803,Canavalia sp.,Climber,42.61,55.70,34%,13.8,15.97697193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Approach 4,CIAT-19032,Canavalia sp.,Climber,65.33,43.83,6%,13.875,16.02639497,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [152]:
data.head()

,subset,requisitioner,no,id_lab,id,n_replicates,tax_order,family,genus,species,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,LMF,2202,F25-0769,AMC-ICARDA-156302,3,Fabales,Fabaceae,Trifolium,repens,...,86.99,6.03,14.08,15.97,16.37,3.75,30.99,-75.10,43.67,72.00
1,1,LMF,2235,F25-0770,AMC-ICARDA-165072,3,NaN,NaN,Vicia,tenuifolia,...,86.99,6.19,14.43,14.37,18.77,2.97,31.76,NaN,56.12,56.78
2,1,LMF,2226,F25-0773,AMC-ICARDA-168125,3,NaN,NaN,Lathyrus,sylvestris,...,61.77,4.70,10.30,15.53,17.77,3.12,22.66,NaN,55.10,42.32
3,1,Genetic bank,1057,F24-3469,CIAT-10647,6,Fabales,Fabaceae,Stylosanthes,scabra,...,79.20,6.90,14.88,15.90,22.28,3.95,32.01,151.04,48.31,67.34
4,1,Genetic bank,1060,F24-3470,CIAT-11194,6,Fabales,Fabaceae,Stylosanthes,hamata,...,70.41,5.14,12.69,15.17,20.65,3.91,27.31,184.66,46.12,59.25


In [153]:
requested.rename(columns={'gene_bank_breeding_id': 'id'}, inplace=True)
requested['id'] = clean_standardize_ids(requested['id'])
approach_id  = requested.iloc[:, :2]
approach_id.head(50)

,approach,id
1,Approach 1 (a),CIAT-772
2,Approach 1 (a),CIAT-19213
3,Approach 2,CIAT-9434
4,Approach 2,CIAT-955
5,Approach 4,CIAT-9432
6,Approach 4,CIAT-18447
7,Approach 1 (a),CIAT-7317
8,Approach 1 (a),CIAT-8719
9,Approach 2,CIAT-20803
10,Approach 4,CIAT-19032


In [154]:
merged_df = approach_id.merge(data, on='id')
merged_df.head(50)

,approach,id,subset,requisitioner,no,id_lab,n_replicates,tax_order,family,genus,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,Approach 1 (a),CIAT-772,1,Genetic bank,18,F24-3421,9,Fabales,Fabaceae,Clitoria,...,75.07,4.33,11.18,13.46,16.11,4.06,24.42,77.57,37.92,65.10
1,Approach 1 (a),CIAT-19213,1,Genetic bank,1649,F24-3501,9,Fabales,Fabaceae,Clitoria,...,71.89,7.58,11.57,14.99,18.66,4.15,24.65,194.95,39.14,63.66
2,Approach 2,CIAT-9434,2,Genetic bank,83,F24-3635,9,Fabales,Fabaceae,Clitoria,...,81.72,6.21,13.39,14.89,17.69,3.03,29.55,3.51,56.61,54.00
3,Approach 2,CIAT-955,1,Genetic bank,32,F24-3425,9,Fabales,Fabaceae,Clitoria,...,84.29,4.83,12.25,12.82,15.90,3.29,26.67,138.42,45.45,59.65
4,Approach 4,CIAT-9432,1,Genetic bank,1039,F24-3463,6,Fabales,Fabaceae,Clitoria,...,70.80,6.57,13.41,17.47,20.60,3.57,29.38,-897.64,54.34,55.33
5,Approach 4,CIAT-9432,3,Breeding,1002,F25-2635,18,Fabales,Fabaceae,Clitoria,...,84.52,7.57,13.71,15.25,17.61,2.72,28.73,100.01,60.36,48.13
6,Approach 4,CIAT-18447,1,Genetic bank,1218,F24-3484,9,Fabales,Fabaceae,Clitoria,...,80.39,6.57,13.64,14.51,20.06,3.20,29.34,361.74,53.02,55.28
7,Approach 1 (a),CIAT-7317,1,Genetic bank,185,F24-3435,36,Fabales,Fabaceae,Canavalia,...,64.66,5.20,10.57,14.86,18.14,5.01,22.85,108.22,32.97,69.48
8,Approach 1 (a),CIAT-8719,1,Genetic bank,347,F24-3453,30,Fabales,Fabaceae,Canavalia,...,65.97,5.12,10.94,14.29,18.98,4.97,23.39,239.49,34.19,69.74
9,Approach 2,CIAT-20803,1,Genetic bank,1905,F24-3512,9,Fabales,Fabaceae,Canavalia,...,68.27,4.88,10.92,13.80,18.37,3.86,23.37,344.33,42.61,55.70


In [155]:
merged_df.columns

Index(['approach', 'id', 'subset', 'requisitioner', 'no', 'id_lab',
       'n_replicates', 'tax_order', 'family', 'genus', 'species', 'tax_name',
       'functional_group', 'set_ciat', 'batch', 'run', 'replication',
       'syrange', 'sample_weight_g', 'undigested_dm_g', 'dm_incubated',
       'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml', 'ch4_8h_ml',
       'ch4_24h_ml', 'ch4_8h_percentage', 'ch4_24h_percentage', 'part_fact',
       'ch4_ml_g_dm_incubaed_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm'],
      dtype='object')

In [156]:
primary_traits = merged_df[['approach', 'id','tax_name','functional_group', 'dm_incubated',  'digested_feed_mg', 'ch4_24h_ml']]
primary_traits.head(50)

,approach,id,tax_name,functional_group,dm_incubated,digested_feed_mg,ch4_24h_ml
0,Approach 1 (a),CIAT-772,Clitoria ternatea,Herbaceous_legumes,457.74,298.00,11.18
1,Approach 1 (a),CIAT-19213,Clitoria ternatea,Herbaceous_legumes,469.33,298.75,11.57
2,Approach 2,CIAT-9434,Clitoria ternatea,Herbaceous_legumes,454.10,236.60,13.39
3,Approach 2,CIAT-955,Clitoria ternatea,Herbaceous_legumes,459.34,274.00,12.25
4,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,456.56,252.63,13.41
5,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,477.33,229.65,13.71
6,Approach 4,CIAT-18447,Clitoria ternatea,Herbaceous_legumes,464.97,257.04,13.64
7,Approach 1 (a),CIAT-7317,Canavalia sp.,Herbaceous_legumes,462.50,321.35,10.57
8,Approach 1 (a),CIAT-8719,Canavalia sp.,Herbaceous_legumes,467.83,326.25,10.94
9,Approach 2,CIAT-20803,Canavalia sp.,Herbaceous_legumes,467.17,260.21,10.92


In [157]:
#primary_traits.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/primary_traits_to_iomicas.csv', index=None)

# 5.0 Grasses for Breeding

Requested by: Claudia Perea

Date: 2026_05_26

## 5.1 Gas Data Processing for Breeding

### 5.1.1 Gas Data Loading

In [158]:
df = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/05_grasses_for_breeding/gas_clean_subsets_1234_2026_06_09.csv')
df.head(2)


,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2,delete
0,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,0,95.43754089,34.829738,Standar,NaN,NaN,NaN,NaN,NaN,no
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,0,89.75200777,36.194699,Standar,NaN,NaN,NaN,NaN,NaN,no


### 5.1.2 Filtering and Formatting Gas Data

In [159]:
df.functional_group.unique()

array(['Forage', 'Herbaceous_legumes', 'Grass', 'Shrub_Trees', nan,
       'Browse', 'Concentrate'], dtype=object)

In [160]:
df.requisitioner.unique()

array(['LMF', 'Genetic_bank', 'Breeding', 'LMF-invivo', 'Benchmark',
       'Isabel Molina', 'Mauricio Sotelo',
       'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [161]:
df1 = df[df['functional_group'] == 'Grass']

In [162]:
df1.to_csv('grass.csv', index=None)

In [163]:
df2 = df1[df1['requisitioner'].isin(['Breeding', 'Benchmark'])]

In [164]:
df2.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [165]:
df3 = df2[['subset','no','requisitioner','id_lab','id','tax_name','functional_group',
           'batch','run','replication', 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h'
       ]]
df3 = df3.copy()

cols = [ 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

df3[cols] = df3[cols].apply(pd.to_numeric, errors='coerce')
df3[cols] = df3[cols].round(2)

In [166]:
df3.tail()

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,batch,run,replication,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
5903,4,1144,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,65,2,2,32.86,86.98,177.11,45.82,2.59,13.0,14.31,25.34,55.30,0.47
5964,4,1213,Breeding,F25-2104,CIAT-BR-06-1348,Urochloa interespecific,Grass,71,3,1,42.53,95.72,205.57,54.04,2.63,13.8,14.91,30.65,56.72,0.62
5965,4,1214,Breeding,F25-2104,CIAT-BR-06-1348,Urochloa interespecific,Grass,71,3,2,38.03,92.72,199.04,69.56,3.49,13.4,14.70,29.25,42.06,0.28
5966,4,1215,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,71,3,1,35.53,90.22,183.54,51.32,2.80,14.1,15.19,27.88,54.33,0.41
5967,4,1216,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,71,3,2,35.03,88.22,179.44,38.84,2.16,13.4,14.79,26.53,68.32,0.77


In [167]:
#df3.to_csv('breeding.csv', index=None)

In [168]:
df3.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/gas_breeding_complete.csv', index=None)

### 5.1.3 Average Gas Values

In [169]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_name', 'functional_group', 'batch',
       'run', 'replication']
numeric_columns = [ 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

for df in [df3]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [170]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [171]:
df4 = mean_with_replicates(df3, category_columns, numeric_columns)
df4.tail(2)

/tmp/ipykernel_42361/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_42361/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_42361/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


,id_lab,subset,no,requisitioner,id,tax_name,functional_group,batch,run,replication,...,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
179,F25-2627,3,978,Benchmark,BR-02-1752-Cayman-Exc,Urochloa interespecific,Grass,55,1,1,...,89.08,184.24,53.97,2.93,13.85,14.78,27.23,50.57,244.48,6
180,F25-2628,3,981,Benchmark,BR-06-423-Cayman-Exc,Urochloa interespecific,Grass,55,1,1,...,91.99,190.23,57.93,3.06,13.47,14.80,28.16,48.72,235.63,6


In [172]:
df4

,id_lab,subset,no,requisitioner,id,tax_name,functional_group,batch,run,replication,...,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
0,F24-3582,1,226,Breeding,CIAT-BR-02-1752,Urochloa interespecific,Grass,10,1,1,...,74.98,156.06,67.39,4.32,13.98,14.61,22.79,33.83,100.28,9
1,F24-3583,1,229,Breeding,CIAT-BR-02-1794,Urochloa interespecific,Grass,10,1,1,...,76.60,157.75,63.97,4.06,13.79,14.51,22.88,35.77,116.14,9
2,F24-3584,1,232,Breeding,CIAT-BR-06-0423,Urochloa interespecific,Grass,10,1,1,...,73.82,152.46,64.27,4.22,14.61,15.14,23.12,35.98,108.27,8
3,F24-3585,1,235,Breeding,CIAT-BR-09-1232,Urochloa interespecific,Grass,10,1,1,...,81.29,167.08,66.64,3.99,13.76,14.37,24.02,36.04,108.20,8
4,F24-3586,1,238,Breeding,CIAT-BR-12-4951,Urochloa interespecific,Grass,10,1,1,...,75.13,156.86,65.93,4.21,14.34,15.02,23.59,35.76,103.45,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,F25-2624,3,969,Benchmark,CIAT-6294-Marandu-Exc,Urochloa interespecific,Grass,55,1,1,...,97.99,203.60,56.67,2.79,13.92,14.98,30.52,53.97,259.75,6
177,F25-2625,3,1080,Benchmark,CIAT-36087-MulatoII-Exc,Urochloa interespecific,Grass,55,2,1,...,92.73,190.98,57.70,3.02,13.33,14.29,27.30,47.33,229.78,3
178,F25-2626,3,975,Benchmark,CIAT-606-Basilisk-Exc,Urochloa interespecific,Grass,55,1,1,...,100.99,208.39,57.44,2.76,13.63,14.55,30.33,52.82,255.97,6
179,F25-2627,3,978,Benchmark,BR-02-1752-Cayman-Exc,Urochloa interespecific,Grass,55,1,1,...,89.08,184.24,53.97,2.93,13.85,14.78,27.23,50.57,244.48,6


In [173]:
df4.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/gas_breeding_average.csv', index=None)

## 5.2. Nutrition Data Processing for Breeding

### 5.2.1 Nutrition Data Loading

In [174]:
nu = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_complete_1234_2026_06_09.csv')
nu.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic_bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic_bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


### 5.2.2 Filtering and Formatting Nutrition Data

In [175]:
nu.functional_group.unique()

array(['Herbaceous_legumes', 'Shrub_Trees', 'Grass', 'Browse', 'Forage',
       'Concentrate', nan], dtype=object)

In [176]:
nu.requisitioner.unique()

array(['Genetic_bank', 'Breeding', 'LMF', 'LMF-invivo', 'Benchmark',
       'Isabel Molina', 'Mauricio Sotelo',
       'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [177]:
nu1 = nu[nu['functional_group'] == 'Grass']

In [178]:
nu2 = nu1[nu1['requisitioner'].isin(['Breeding', 'Benchmark'])]
nu2.tail(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
1910,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Poales,Poaceae,Urochloa,humidicola,Urochloa humidicola,Grass,30,98.20,12.48,87.52,8.80,36.63,73.37
1911,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Poales,Poaceae,Urochloa,humidicola,Urochloa humidicola,Grass,30,98.22,15.08,84.92,8.73,36.96,73.74


In [179]:
cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']
nu2[cols] = nu2[cols].round(2)

/tmp/ipykernel_42361/1946393332.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nu2[cols] = nu2[cols].round(2)


In [180]:
#nu2.to_csv('nutrition.csv', index=None)

In [181]:
#nu2.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_breeding_complete.csv', index=None)

In [182]:
nu2.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [183]:
nu3 = nu2[['subset','no','requisitioner','id_lab','id','tax_name','functional_group','dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']]
nu3 = nu3.copy()

cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']
nu3[cols] = nu3[cols].round(2)

In [184]:
nu3

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
52,1,51.0,Breeding,F24-3582,CIAT-BR-02-1752,Urochloa interespecific,Grass,95.98,15.64,84.36,10.92,21.20,55.59
53,1,52.0,Breeding,F24-3582,CIAT-BR-02-1752,Urochloa interespecific,Grass,96.07,15.76,84.24,10.92,21.09,55.23
54,1,53.0,Breeding,F24-3583,CIAT-BR-02-1794,Urochloa interespecific,Grass,96.97,15.14,84.86,9.39,21.27,54.83
55,1,54.0,Breeding,F24-3583,CIAT-BR-02-1794,Urochloa interespecific,Grass,97.12,14.90,85.10,9.39,22.40,56.70
56,1,55.0,Breeding,F24-3584,CIAT-BR-06-0423,Urochloa interespecific,Grass,96.56,15.97,84.03,11.01,22.15,57.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1521,3,212.0,Benchmark,F25-2628,BR-06-423-Cayman-Exc,Urochloa interespecific,Grass,96.77,10.43,89.57,13.82,29.08,61.44
1908,4,NaN,Breeding,F25-2104,CIAT-BR-06-1348,Urochloa interespecific,Grass,93.02,14.27,85.73,12.27,25.43,59.24
1909,4,NaN,Breeding,F25-2104,CIAT-BR-06-1348,Urochloa interespecific,Grass,93.16,14.68,85.32,12.18,25.85,60.28
1910,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,98.20,12.48,87.52,8.80,36.63,73.37


### 5.2.3 Average Nutrition Values

In [185]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_name', 'functional_group']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']

for df in [nu3]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [186]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [187]:
nu4 = mean_with_replicates(nu3, category_columns, numeric_columns)
nu4.tail(2)

/tmp/ipykernel_42361/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_42361/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_42361/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


,id_lab,subset,no,requisitioner,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
179,F25-2627,3,209.0,Benchmark,BR-02-1752-Cayman-Exc,Urochloa interespecific,Grass,96.64,17.21,82.79,10.93,29.65,59.76,2
180,F25-2628,3,211.0,Benchmark,BR-06-423-Cayman-Exc,Urochloa interespecific,Grass,96.67,10.42,89.57,13.86,29.45,61.60,2


In [188]:
nu4.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_breeding_average.csv', index=None)

## 5.3 Compilation of Gas and Nutrition Data

In [189]:
df4.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_name',
       'functional_group', 'batch', 'run', 'replication', 'net_gas_8h_ml',
       'net_gas_24h_ml', 'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas'],
      dtype='object')

In [190]:
df5 = df4[['id_lab','net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [191]:
compiled = nu4.merge(df5, on='id_lab', how='left')

In [192]:
#compiled.to_csv('compiled.csv', index=None)

In [193]:
# Save the compiled gas and nutrition breeding average data
compiled.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/compilated_gas_nutrition_breeding_average.csv', index=None)

## 5.4 Dataset for BLUES and BLUPs

In [194]:
df3.head(2) # Filtered dataset for Breeding

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,batch,run,replication,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
199,1,226,Breeding,F24-3582,CIAT-BR-02-1752,Urochloa interespecific,Grass,10,1,1,36.65,73.60,153.24,66.86,4.36,13.6,14.35,21.99,32.90,98.77
200,1,227,Breeding,F24-3582,CIAT-BR-02-1752,Urochloa interespecific,Grass,10,1,2,36.65,70.51,146.73,65.35,4.45,13.7,14.32,21.02,32.16,101.24


In [195]:
df3.to_csv('df3.csv', index=None)

In [196]:
star = df[df['id_lab'] == 'F25-0019']
star.head(2)

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm


In [197]:
star.batch.unique()

AttributeError: 'DataFrame' object has no attribute 'batch'

In [ ]:
star_in_batch = star[star["batch"].isin(batches)]
star_in_batch.batch.unique()

In [ ]:
star3 = star_in_batch[['subset','no','requisitioner','id_lab','id','tax_name','functional_group',
           'batch','run','replication', 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h'
       ]]
star3 = star3.copy()

cols = [ 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

star3[cols] = star3[cols].apply(pd.to_numeric, errors='coerce')
star3[cols] = star3[cols].round(2)

In [ ]:
star3.batch.unique()

In [ ]:
df3.batch.unique()

In [ ]:
breeding_and_star = pd.concat([df3, star3])

In [ ]:
#breeding_and_star.to_csv('gas_breeding_plus_stargrass.csv', index = None)

# 6.0 Stylosanthes Gene Bank

Requested by: Juan José González

Date: 2026_05_25

## 6.1 Data Loading for Stylosanthes Gene Bank

In [ ]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/data_requested_juan_jose.csv')
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/gas_clean_subsets_1234_2026_06_09.csv')
nu = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/nutrition_complete_1234_2026_06_09.csv')

## 6.2 Filtering and Formatting Stylosanthes Gene Bank Data

In [ ]:
requested['id'] = clean_standardize_ids(requested['id'])

In [ ]:
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'set_ciat','functional_group','batch','run','replication','syrange']
numeric_columns = ['net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

for df in [gas, nu]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
cols = [
    'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h'
]

gas2 = gas[['id'] + cols].copy()


gas2[cols] = gas2[cols].round(2)


gas2.head(2)

In [ ]:
nu2 = nu[['subset', 'no', 'requisitioner', 'id_lab', 'id', 'functional_group',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']]
nu2.round(2)
cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']
nu2[cols] = nu2[cols].round(2)
nu2.head(2)

## 6.3 Compilation of Stylosanthes Gene Bank Data

In [ ]:
requested_gas = requested.merge(gas2, on='id', how='left')

In [ ]:
requested_gas

In [ ]:
requested_nutrition = requested.merge(nu2, on='id', how='left')

In [ ]:
requested_gas

In [ ]:
requested_nutrition

In [ ]:
requested_gas.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/06_stylosanthes_gene_bank/stylosanthes_gas_genebank.csv', index =None)

In [ ]:
requested_nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/06_stylosanthes_gene_bank/stylosanthes_nutrition_genebank.csv', index = None)

# 7.0  NIRs Compilation

---

Requested by: Juan Andrés

Date: 2026_06

## 7.1 Data load

In [ ]:
gas_av = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/gas_clean_average_subsets_1234_2026_06_09.csv')
nu_av = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/nutrition_average_1234_2026_06_09.csv')

ggb = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/grass_genebank.csv')
gt = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/total_grass.csv')
leg = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/legumes.csv')


In [ ]:
gt

In [ ]:
nu_av.head(5)

## 7.2 Filtering and formatting

In [ ]:
gt['id_lab'] = clean_lab_ids(gt['id_lab'])
ggb['id_lab'] = clean_lab_ids(ggb['id_lab'])
leg['id_lab'] = clean_lab_ids(leg['id_lab'])

gt['id'] = clean_standardize_ids(gt['id'])
ggb['id'] = clean_standardize_ids(ggb['id'])
leg['id'] = clean_standardize_ids(leg['id'])

## 7.3 Compilated average values for Gas and Nutrition





In [ ]:
gas_av.columns

In [ ]:
nu_av.columns

In [ ]:
nu_av2 = nu_av[['id_lab', 'id', 'tax_name', 'functional_group', 'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']]
nu_av2.head(2)

In [ ]:
gas_av2 = gas_av[['id_lab','net_gas_8h_ml', 'net_gas_24h_ml',
       'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h']]
gas_av2.head(2)

In [ ]:
compiled = nu_av2.merge(gas_av2, on='id_lab', how='left')
compiled.head(2)

## 7.4 Traits and NIRs data frame creation

### 7.4.1 Grasses total


In [ ]:
gt.columns

In [ ]:
requested_gt = gt.id_lab.unique()
print('Requested length:',len(requested_gt))
print(requested_gt)

In [ ]:
gt_2 = gt.drop(['position', 'id', 'ot_lab', 'tax_name', 'functional_group'], axis=1)
gt_2.head(2)

In [ ]:
requested_compiled = compiled[compiled['id_lab'].isin(requested_gt)]
print('requested_compiled lenght:', len(requested_compiled))
requested_compiled.head(2)

In [ ]:
gt_traits_nirs = requested_compiled.merge(gt_2, on='id_lab', how='left')
print('gt_traits_nirs lenght:', len(gt_traits_nirs))
gt_traits_nirs.head(2)

In [ ]:
#gt_traits_nirs.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/07_nirs_compilation/total_grass_nirs.csv', index = None)

### 7.4.2 Grasses for Gene Bank

In [ ]:
ggb_list = ggb.id_lab.unique()
print('Requested list length:',len(ggb_list))

compiled_list = compiled.id_lab.unique()
print('Compiled list length:',len(compiled_list))

commons = set(ggb_list) & set(compiled_list)
print('Commons list length:',len(commons))
print(commons)

According with this comparison, no samples are shared between requested and compiled lists

### 7.4.3 Legumes

In [ ]:
leg.head(1)

In [ ]:
leg_2 = leg.drop(['position', 'id', 'ot_lab', 'tax_name', 'functional_group'], axis=1)
leg_2.head(1)

In [ ]:
requested_list = leg_2.id_lab.unique()
print('Requested list length:',len(requested_list))

compiled_list = compiled.id_lab.unique()
print('Compiled list length:',len(compiled_list))

commons = set(requested_list) & set(compiled_list)
print('Commons list length:',len(commons))
print(commons)

In [ ]:
commons_traits =  compiled[compiled['id_lab'].isin(commons)].copy()
print('commons_traits lenght:', len(commons_traits))
commons_traits.tail()

In [ ]:
legumes_traits_nirs = commons_traits.merge(leg_2, on='id_lab', how='left')
print('legumes_traits_nirs lenght:', len(legumes_traits_nirs))
legumes_traits_nirs.head(2)

In [ ]:
legumes_traits_nirs.iloc[:2, 18:25]

In [ ]:
legumes_traits_nirs_clean = legumes_traits_nirs.drop(columns=legumes_traits_nirs.columns[legumes_traits_nirs.columns.str.contains('Unnamed')])

In [ ]:
legumes_traits_nirs_clean.head(2)

In [ ]:
#legumes_traits_nirs_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/07_nirs_compilation/legumes_nirs.csv', index = None)

# 8.0 Dashboard June 2026

Requested by: Alejandara Marín

Date: 2026_06_22

## 8.1. Data load

In [198]:
gas_av = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/08_dashboard_june_2026/gas_clean_average_subsets_1234_2026_06_09.csv')

## 8.2 Filtering

In [199]:
gas_av.columns

Index(['id_lab', 'id', 'subset', 'no', 'requisitioner', 'tax_name',
       'functional_group', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas'],
      dtype='object')

In [200]:
gas_av2 = gas_av[['requisitioner','n_replicates_gas','subset','id','tax_name','functional_group', 'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h','methane_intensity','tddm']]
gas_av2.head(2)

,requisitioner,n_replicates_gas,subset,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,LMF-invivo,6,3,Dieta-1-Exp-1,Dieta 1_Exp1,NaN,15.67,15.45,81.06,29.35
1,LMF-invivo,5,3,Dieta-1-Exp-2,Dieta 1_Exp2,NaN,14.18,14.74,76.94,17.76


In [201]:
gas_av3 = gas_av2.sort_values(by=['subset', 'id'])

In [202]:
gas_av3.functional_group.unique()

array(['Herbaceous_legumes', nan, 'Shrub_Trees', 'Browse', 'Grass',
       'Forage', 'Concentrate'], dtype=object)

In [203]:
grass = gas_av3[gas_av3['functional_group'] == 'Grass']
shrub = gas_av3[gas_av3['functional_group'] == 'Shrub_Trees']
legumes = gas_av3[gas_av3['functional_group'] == 'Herbaceous_legumes']

In [204]:
grass.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/grass.csv', index = None)
shrub.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/shrub_trees.csv', index = None)
legumes.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/legumes.csv', index = None)

In [205]:
gas_av3.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/gas.csv', index = None)

## 8.3 Sort dataframes by tddm and methane intensity

In [219]:
grass_sorted = grass.sort_values(by=["tddm", "methane_intensity"],  ascending=[False, True]).reset_index(drop=True)
shrub_sorted = shrub.sort_values(by=["tddm", "methane_intensity"],  ascending=[False, True]).reset_index(drop=True)
legumes_sorted = legumes.sort_values(by=["tddm", "methane_intensity"],  ascending=[False, True]).reset_index(drop=True)

In [222]:
grass_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/grass_sorted.csv')
shrub_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/shrub_trees_sorted.csv')
legumes_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/legumes_sorted.csv')